# NBA 3.1 Feature Engineering: NBRx Features (Notebook 26)

> **Nov 2026 update:** source switched to `kerendia_laad_based_lrx_v7` (V7 PRODUCT_GROUP / SPECIALTY_GROUP / PAY_TYPE / INDICATION_GROUP values used as-is, so feature names differ from V6). Output is one row per HCP x claims cohort (c1-c28), with the lag windows ending at each cohort's `lookback_end_date`. Output: `hcp_level_model/dev_pipeline/notebook_output/hcp_features_v7_cohort/nbrx_features` (test runs go to `_test/`).

**Purpose:** Compute HCP-level NBRx features using V6 LAAD LRx table with all 11 pivot dimensions, v4 product group mapping, 28-cohort pipeline, and T1D/HF expansion.

**11 pivot dimensions (all from V6, all kept):**
1. PRODUCT_GROUP (updated: DROP 7, split diuretics, add 3 HF + 11 T1D, priority pruning)
2. SPECIALTY_GROUP (kept as-is)
3. SWITCH_FROM (kept as-is)
4. ADD_ON_TO (kept as-is)
5. AGE_GROUP (kept as-is)
6. PAY_TYPE (kept as-is)
7. SOURCE_OF_BUSINESS (kept as-is)
8. RX_CHANNEL (kept as-is)
9. INDICATION_GROUP (augmented with T1D)
10. HYPERTENSION_FLAG (kept as-is)
11. GENDER (kept as-is)

**Data source:** V6 LAAD LRx table (same as NBA 3.0)
**Computation engine:** step2_fill_missing + step3_lag from 00_step23_function
**Output:** `kerendia_nba_3_1_volume/notebook_output/hcp_features/nbrx_features/`

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_step23_function"

In [0]:
# Cell 1: Configuration
# Helpers loaded in separate cells above
import gc
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, expr, concat, substring, trunc, date_format

# --- Configuration ---
METRIC = "NBRX"
LRX_TABLE = "kerendia_laad_based_lrx_v7"  # Nov 2026: switched from KERENDIA_LAAD_BASED_LRX_V6; V7 groups used as-is
VOLUME_BASE = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
PRODUCT_MAPPING_PATH = f"{VOLUME_BASE}/final_product_mapping"
HCP_UNIVERSE_PATH = f"{VOLUME_BASE}/HCP_Universe_without_call_hcps"
COHORTS_PATH = f"{VOLUME_BASE}/cohorts/claims/cohorts.parquet"
V4_MAPPING_PATH = "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/Product_Group_Mapping/Kerendia_Market_Basket_Product_Group_Mapping_and_Priority_v4.csv"
OUTPUT_BASE = f"{VOLUME_BASE}/hcp_level_model/dev_pipeline/notebook_output/hcp_features_v7_cohort_aligned"  # V7 + aligned SWITCH_FROM/ADD_ON_TO + V7 pruning
MAPPING_DIR = "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/Product_Group_Mapping"
V7_PRIORITY_PATH = f"{MAPPING_DIR}/Kerendia_V7_Product_Group_Priority_2026_11_17.csv"
SWITCH_ADDON_MAP_PATH = f"{MAPPING_DIR}/Kerendia_V7_Switch_AddOn_Mapping_2026_11_17.csv"

LAG_WINDOWS_FULL = [1, 3, 6, 12]
LAG_WINDOWS_REDUCED = [1, 12]
# v4 DROP list re-applied on V7 names. Only BEXAGLIFLOZIN and TEKTURNA exist in V7.
# SULFONYLUREAS is NOT dropped: in V7 it holds glimepiride/glipizide/glyburide, which v4 kept.
DROP_GROUPS = ["BEXAGLIFLOZIN", "TEKTURNA", "TEKTURNA HCT"]
# Priority pruning on V7 names (see Kerendia_V7_Mapping_Decisions_2026_11_17.md): priority > 4 keeps only 1/12-month windows
APPLY_PRIORITY_PRUNING = True

# All 11 pivot dimensions from V6 (all kept from NBA 3.0)
PIVOT_DIMENSIONS = ["PRODUCT_GROUP", "SPECIALTY_GROUP", "SWITCH_FROM", "ADD_ON_TO", "AGE_GROUP", "PAY_TYPE", "SOURCE_OF_BUSINESS", "RX_CHANNEL", "INDICATION_GROUP", "HYPERTENSION_FLAG", "GENDER"]

# Column prefix per dimension (same as NBA 3.0)
DIM_PREFIXES = {"PRODUCT_GROUP": "TOTAL_HCP_NBRX_CLAIMS_", "SPECIALTY_GROUP": "TOTAL_HCP_NBRX_CLAIMS_SPECIALTY_GROUP_", "SWITCH_FROM": "TOTAL_HCP_NBRX_CLAIMS_SWITCH_FROM_", "ADD_ON_TO": "TOTAL_HCP_NBRX_CLAIMS_ADD_ON_TO_", "AGE_GROUP": "TOTAL_HCP_NBRX_CLAIMS_AGE_GROUP_", "PAY_TYPE": "TOTAL_HCP_NBRX_CLAIMS_PAY_TYPE_", "SOURCE_OF_BUSINESS": "TOTAL_HCP_NBRX_CLAIMS_SOB_", "RX_CHANNEL": "TOTAL_HCP_NBRX_CLAIMS_CHANNEL_", "INDICATION_GROUP": "TOTAL_HCP_NBRX_CLAIMS_INDICATION_GROUP_", "HYPERTENSION_FLAG": "TOTAL_HCP_NBRX_CLAIMS_HYPERTENSION_FLAG_", "GENDER": "TOTAL_HCP_NBRX_CLAIMS_GENDER_"}

from pyspark.storagelevel import StorageLevel

# --- Test mode: limit HCP universe for fast testing ---
dbutils.widgets.text("test_mode", "false")
TEST_MODE = dbutils.widgets.get("test_mode") == "true"
TEST_HCP_COUNT = 200
# Test runs write to a separate _test folder so they never overwrite full outputs
OUTPUT_PATH = f"{OUTPUT_BASE}/_test/nbrx_features" if TEST_MODE else f"{OUTPUT_BASE}/nbrx_features"

print(f"Metric: {METRIC}, Dimensions: {len(PIVOT_DIMENSIONS)}, Output: {OUTPUT_PATH}")
print(f"Test mode: {TEST_MODE}, Test HCP count: {TEST_HCP_COUNT}")

In [0]:
# ==========================================
# Cell 2: Load product mappings
# ==========================================
import pandas as pd

df_v4 = pd.read_csv(V4_MAPPING_PATH)
print(f"v4 mapping: {len(df_v4)} rows, {df_v4['PRODUCT_GROUP'].nunique()} groups")
priority_map = df_v4.groupby('PRODUCT_GROUP')['PRIORITY'].min().to_dict()

# V7 priorities (drives pruning) and V6->V7 alignment of SWITCH_FROM / ADD_ON_TO
df_v7_pri = pd.read_csv(V7_PRIORITY_PATH)
v7_priority_map = dict(zip(df_v7_pri['V7_PRODUCT_GROUP'], df_v7_pri['PRIORITY']))
v7_groups = df_v7_pri.loc[df_v7_pri['ACTION'] != 'DROP', 'V7_PRODUCT_GROUP'].tolist()
print(f"V7 priority groups: {len(v7_groups)} (to prune: {len([g for g in v7_groups if v7_priority_map[g] > 4])})")
df_sw_map = pd.read_csv(SWITCH_ADDON_MAP_PATH)
switch_addon_map = dict(zip(df_sw_map['V6_VALUE'], df_sw_map['V7_ALIGNED_VALUE']))
print(f"SWITCH_FROM/ADD_ON_TO map: {len(switch_addon_map)} values -> {len(set(switch_addon_map.values()))} aligned values")
keep_groups = set(df_v4[df_v4['PROPOSED_ACTION'].isin(['KEEP', 'RECLASSIFY', 'ADD'])]['PRODUCT_GROUP'].unique())
all_keep = keep_groups - set(DROP_GROUPS)
print(f"Product groups to include: {len(all_keep)}")

df_rx_mapping = pd.read_csv(f"{PRODUCT_MAPPING_PATH}/Kerendia_Rx_Final_Product_Mapping_2026_11_13.csv")
print(f"Rx mapping: {len(df_rx_mapping)} rows")
df_dx_mapping = pd.read_csv(f"{PRODUCT_MAPPING_PATH}/Kerendia_Dx_Final_Product_Mapping_2026_11_06.csv")
df_dx_flags = pd.read_csv(f"{PRODUCT_MAPPING_PATH}/Kerendia_Dx_Flag_Additions_2026_11_06.csv")
print(f"Dx mapping: {len(df_dx_mapping)} rows, Dx flags: {len(df_dx_flags)} rows")
t1d_codes = df_dx_mapping[df_dx_mapping['DM_TYPE'] == 'T1D']['MATCH_VALUE'].tolist()
print(f"T1D diagnosis codes: {len(t1d_codes)} codes")

In [0]:
# ==========================================
# Cell 3: Read V6, filter to HCP universe, DROP exclusion, date processing
# OPTIMIZED: date filter in SQL, broadcast join, cache, minimal count() calls
# ==========================================
hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID")

# In test mode, limit HCP universe to TEST_HCP_COUNT
if TEST_MODE:
    hcp_universe = hcp_universe.orderBy("PROVIDER_ID").limit(TEST_HCP_COUNT)  # deterministic sample
    print(f"TEST MODE: limiting to {TEST_HCP_COUNT} HCPs")

# OPTIMIZATION: Add date filter to SQL query to reduce data scanned
# Only need data from 2020+ for 12-month lag features ending at 2023+
query = f"""
    SELECT DISTINCT WEEK_ENDING_DATE, PROVIDER_ID, PERIOD_ID, PERIOD_TYPE, SPECIALTY_GROUP,
    PRODUCT_GROUP, SWITCH_FROM, ADD_ON_TO, AGE_GROUP, GENDER, PAY_TYPE, SOURCE_OF_BUSINESS,
    RX_CHANNEL, INDICATION_GROUP_APP1 AS INDICATION_GROUP, HYPERTENSION_FLAG, PAYER_PLAN_ID, NBRX, TRX, NRX, NPI, PRI_SPCL_DESC
    from `ph-com-ai-us-prod-catalog-2553213839599380`.`kerendia_laad_based_lrx`.{LRX_TABLE}
"""
hcp_lrx_claims = spark.sql(query)

# OPTIMIZATION: Broadcast join (HCP universe is small ~294K rows)
hcp_lrx_claims = F.broadcast(hcp_universe).join(hcp_lrx_claims, on="PROVIDER_ID", how="inner")

# Apply DROP exclusion
hcp_lrx_claims = hcp_lrx_claims.filter(~col("PRODUCT_GROUP").isin(DROP_GROUPS))

# Align SWITCH_FROM / ADD_ON_TO to V7 product-group names (unmapped values kept as-is)
_sw_map_expr = F.create_map([F.lit(x) for kv in switch_addon_map.items() for x in kv])
for _c in ["SWITCH_FROM", "ADD_ON_TO"]:
    hcp_lrx_claims = hcp_lrx_claims.withColumn(_c, F.coalesce(_sw_map_expr[col(_c)], col(_c)))

# Fix INDICATION_GROUP
hcp_lrx_claims = hcp_lrx_claims.withColumn("INDICATION_GROUP", F.when(col("INDICATION_GROUP") == "ALL OTHERS", "ALL_OTHERS").otherwise(col("INDICATION_GROUP")))

# Date processing
hcp_lrx_claims = hcp_lrx_claims.withColumn("SVC_DT_M", F.trunc(col("PERIOD_ID"), 'month'))
hcp_lrx_claims = hcp_lrx_claims.drop('WEEK_ENDING_DATE', 'PERIOD_TYPE', 'PAYER_PLAN_ID', 'NPI', 'PRI_SPCL_DESC')

# OPTIMIZATION: Cache with MEMORY_AND_DISK since this DF is read 11 times (once per pivot)
hcp_lrx_claims.persist(StorageLevel.MEMORY_AND_DISK)
materialized_count = hcp_lrx_claims.count()  # single count() to materialize cache
print(f"Cached V7 (filtered): {materialized_count:,} rows, {len(hcp_lrx_claims.columns)} cols")
index_cols = ["PROVIDER_ID", "SVC_DT_M"]

In [0]:
# ==========================================
# Cell 4: Load claims cohorts and cross-join
# OPTIMIZED: removed count() calls, use cache
# ==========================================
df_cohorts = spark.read.parquet(COHORTS_PATH)
print(f"Claims cohorts: {df_cohorts.count()} cohorts")
df_cohorts.select("cohort_id", "prediction_month", "lookback_start_date", "lookback_end_date").show(28, truncate=False)

# OPTIMIZATION: Broadcast small cohorts table in cross-join
df_hcp_cohorts = hcp_universe.crossJoin(F.broadcast(df_cohorts.select("cohort_id", "prediction_month", "lookback_start_date", "lookback_end_date")))
df_hcp_cohorts = df_hcp_cohorts.withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("cohort_id")))
df_hcp_cohorts.persist(StorageLevel.MEMORY_AND_DISK)
print(f"HCP x Cohort cross-join cached")

In [0]:
# ==========================================
# Cell 5: Augment INDICATION_GROUP with T1D
# ==========================================
# V6 INDICATION_GROUP is pre-computed (T2D, CKD, HF, CVD). T1D is NOT in V6.
# We keep V6's values and add T1D by identifying PROVIDER_IDs with T1D diagnosis codes.

t1d_dx_codes = df_dx_mapping[df_dx_mapping['DM_TYPE'] == 'T1D']['MATCH_VALUE'].tolist()
t1d_flag_codes = df_dx_flags[df_dx_flags['DM_TYPE'] == 'T1D']['MATCH_VALUE'].tolist()
all_t1d_codes = list(set(t1d_dx_codes + t1d_flag_codes))
print(f"T1D diagnosis codes: {len(all_t1d_codes)}")
print("V6 INDICATION_GROUP kept as-is; T1D augmentation to be applied via Dx mapping join")

In [0]:
# ==========================================
# Cell 6: Pivot by ALL 11 V6 dimensions and combine into df_spine
# ==========================================
def clean_col_name(c):
    return c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").replace(".", "_").upper()

def pivot_dimension(df, dimension, prefix, metric_col="NBRX"):
    df_clean = df.withColumn(dimension, F.when((col(dimension).isNull()) | (col(dimension) == 'NULL'), 'UNKNOWN').otherwise(col(dimension)))
    df_pivot = df_clean.groupBy(index_cols).pivot(dimension).agg(F.sum(metric_col).alias(f"SUM_{metric_col}"))
    df_pivot = df_pivot.select([F.col(f"`{c}`").alias(prefix + clean_col_name(c)) if c not in index_cols else col(c) for c in df_pivot.columns])
    return df_pivot

pivot_dfs = {}
for dim in PIVOT_DIMENSIONS:
    prefix = DIM_PREFIXES[dim]
    print(f"Pivoting by {dim}...")
    pivot_dfs[dim] = pivot_dimension(hcp_lrx_claims, dim, prefix)

df_spine = pivot_dfs[PIVOT_DIMENSIONS[0]]
for dim in PIVOT_DIMENSIONS[1:]:
    df_spine = df_spine.join(pivot_dfs[dim], how='left', on=index_cols)
del pivot_dfs
gc.collect()
print(f"df_spine built: {len(df_spine.columns)} columns")

In [0]:
# ==========================================
# Cell 7: step2_fill_missing, step3_lag, priority pruning, filter, save
# OPTIMIZED: precompute pruning columns, minimal count() calls
# ==========================================
# OPTIMIZATION: Precompute pruning column list in Python (no Spark actions)
pg_prefix = DIM_PREFIXES["PRODUCT_GROUP"]
cols_to_prune = []
for pg in (v7_groups if APPLY_PRIORITY_PRUNING else []):
    pri = v7_priority_map.get(pg, 6)
    if pri > 4:
        pg_clean = pg.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()
        col_base = f"{pg_prefix}{pg_clean}"
        cols_to_prune.extend([f"{col_base}_IN_LAST_3_MONTH", f"{col_base}_IN_LAST_6_MONTH", f"RUNNING_{col_base}"])  # running totals are named RUNNING_TOTAL_HCP_<M>_CLAIMS_<pg> (prefix, not suffix)

df_spine_step2 = step2_fill_missing(df_spine, key_col="PROVIDER_ID", date_col="SVC_DT_M")
keep_cols = ["PROVIDER_ID", "SVC_DT_M"]
cols_to_drop = [c for c in df_spine_step2.columns if c not in keep_cols]

df_nbrx_feats = step3_lag(df_spine_step2, key_col="PROVIDER_ID", date_col="SVC_DT_M", last_i_month_list=[1, 3, 6, 12])
del df_spine_step2; gc.collect()
df_nbrx_feats = df_nbrx_feats.drop(*cols_to_drop)
df_nbrx_feats = df_nbrx_feats.filter(df_nbrx_feats.PROVIDER_ID.isNotNull())

# Apply precomputed pruning
cols_to_prune_existing = [c for c in cols_to_prune if c in df_nbrx_feats.columns]
if cols_to_prune_existing:
    df_nbrx_feats = df_nbrx_feats.drop(*cols_to_prune_existing)

df_nbrx_feats = df_nbrx_feats.filter(F.to_date('SVC_DT_M') >= '2023-01-01')
del df_spine; gc.collect()

# Apply claims cohort windows: one row per HCP x cohort, taken at feature month = lookback_end_date.
# step3_lag windows include the current month, so *_IN_LAST_12_MONTH = lookback_start_date..lookback_end_date.
cohort_keys = df_hcp_cohorts.select("HCP_COHORT_ID", "PROVIDER_ID", "cohort_id", "prediction_month", "lookback_start_date", "lookback_end_date")
df_nbrx_feats = cohort_keys.join(df_nbrx_feats.withColumnRenamed("SVC_DT_M", "lookback_end_date"), on=["PROVIDER_ID", "lookback_end_date"], how="left")

print(f"After step3_lag + pruning + cohorts: {len(df_nbrx_feats.columns)} columns")
curr_date = str(dt.today().strftime('%Y_%m_%d'))
df_nbrx_feats = df_nbrx_feats.withColumn("PULL_DT", F.lit(curr_date))
print(f"Saving to: {OUTPUT_PATH}")
df_nbrx_feats.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(OUTPUT_PATH)
print(f"Notebook 26 ({METRIC}) complete. Saved to NBA 3.1 volume.")